# XLA V4 — P2.F Screening & Confirmation Ablation Runner (`R0` -> `R5-l`)
- **Chế độ:** 100% Offline (`HF_HUB_OFFLINE=1`, `TRANSFORMERS_OFFLINE=1`, `HF_DATASETS_OFFLINE=1`), đường dẫn tuyệt đối `/kaggle/input/...`, Fail-Fast.
- **Gate tự động:** Kiểm tra `Bước 0 ≡ B-Free` (`|Δ logits| < 1e-3`) ngay trên GPU trước mỗi run khởi tạo từ B-Free.
- **Đầu ra mỗi Run `<RUN_ID>`:** `/kaggle/working/p2f_runs/<RUN_ID>/{train_log.csv, scores.parquet, eval_summary.csv, bfree_globalforge_lora_r16_last.pth}`.


In [ ]:
# ============================ CELL 1: CHỌN RUN CẦN CHẠY ============================
# Gợi ý chia phiên Kaggle (mỗi run sàng lọc ~35-40 phút trên 25% data, 3 epochs):
#   Phiên 1 (Recipe & W1):      RUNS_TO_EXECUTE = "R0,R1,R1b"
#   Phiên 2 (Module H-C):       RUNS_TO_EXECUTE = "R2,R3,R4,R5"
#   Phiên 3 (Control P2.B/C/D): RUNS_TO_EXECUTE = "R5-d,R5-a,R5-f,R5-l"
RUNS_TO_EXECUTE = "R1,R1b,R2,R3,R4,R5"
STAGE = "screening"   # "screening" (25% data, 3 epochs) | "confirm" (100% data, 5 epochs)
BATCH_SIZE = 64
SEED = 42
EVAL_MAX_PER_GROUP = 200


In [ ]:
# ============================ CELL 2: OFFLINE ENV & FAIL-FAST PATHS ============================
import os
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "0"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

WHEELS_DIR        = Path("/kaggle/input/notebooks/baopv051/xla-setup/wheels_rtxpro6000")
WHEELS_EXTRA_DIR  = Path("/kaggle/input/notebooks/baopv051/xla-setup-eval/offline_bundle/wheels_eval_extra")
REPO_SRC          = Path("/kaggle/input/notebooks/baopv051/xla-setup/bfree_src")
DINOV2_SD         = Path("/kaggle/input/notebooks/baopv051/xla-setup/models/vit_base_patch14_reg4_dinov2/model.safetensors")
BFREE_WEIGHTS_DIR = Path("/kaggle/input/models/baopv051/bfree-baseline-weights/pytorch/default/1")
TRAIN_DATA_ROOT   = Path("/kaggle/input/datasets/baopv051/b-free-training-data")
AI_GENBENCH_DIR   = Path("/kaggle/input/datasets/baopv051/ai-genbench-v1")
WILDRF_DIR        = Path("/kaggle/input/datasets/ngovanphuong2604/wildrf-dataset")
SYNTHWILDX_DIR    = Path("/kaggle/input/datasets/ngovanphuong2604/synthwildx-dataset")
BFREE_VIRAL_DIR   = Path("/kaggle/input/datasets/ngovanphuong2604/bfree-viral-images")
OUTPUT_ROOT       = Path("/kaggle/working/p2f_runs")

def require_dir(p: Path, what: str) -> Path:
    if not p.is_absolute() or not p.is_dir():
        raise FileNotFoundError(f"FAIL-FAST (missing dir): {what} -> expected {p}")
    return p

require_dir(WHEELS_DIR, "WHEELS_DIR (xla-setup)")
require_dir(REPO_SRC / "code", "REPO_SRC/code (xla-setup)")
require_dir(BFREE_WEIGHTS_DIR, "BFREE_WEIGHTS_DIR")
require_dir(TRAIN_DATA_ROOT, "TRAIN_DATA_ROOT")
require_dir(AI_GENBENCH_DIR, "AI_GENBENCH_DIR")
print("[OK] All required Kaggle input directories verified.")


In [ ]:
# ============================ CELL 3: OFFLINE WHEEL INSTALL & VERSION GATE ============================
import subprocess, sys

find_links = ["--find-links", str(WHEELS_DIR)]
if WHEELS_EXTRA_DIR.is_dir():
    find_links += ["--find-links", str(WHEELS_EXTRA_DIR)]

pkgs = [
    "torch==2.8.0", "torchvision==0.23.0", "timm==1.0.22", "peft==0.15.2",
    "transformers==4.55.4", "huggingface_hub==0.36.0", "tokenizers==0.21.1",
    "accelerate==1.3.0", "pandas==2.3.3", "safetensors", "numpy", "scipy",
    "scikit-learn", "pyyaml", "pillow", "tqdm", "pyarrow", "datasets"
]
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *find_links, *pkgs], check=False)

import torch, timm, peft
assert torch.cuda.is_available(), "FAIL-FAST: CUDA GPU not enabled!"
print(f"[OK] torch={torch.__version__} | timm={timm.__version__} | peft={peft.__version__} | GPU={torch.cuda.get_device_name(0)}")


In [ ]:
# ============================ CELL 4: CHẠY LƯỚI ABLATION P2.F ============================
import subprocess, sys

runner_script = REPO_SRC / "code" / "scripts" / "run_p2f_ablation_kaggle.py"
assert runner_script.is_file(), f"Missing {runner_script} — push latest branch & re-run xla-setup!"

cmd = [
    sys.executable, str(runner_script),
    "--runs", RUNS_TO_EXECUTE,
    "--stage", STAGE,
    "--batch_size", str(BATCH_SIZE),
    "--seed", str(SEED),
    "--eval_max_per_group", str(EVAL_MAX_PER_GROUP),
    "--repo_src", str(REPO_SRC),
    "--wheels_dir", str(WHEELS_DIR),
    "--dinov2_sd", str(DINOV2_SD),
    "--bfree_weights_dir", str(BFREE_WEIGHTS_DIR),
    "--train_data_root", str(TRAIN_DATA_ROOT),
    "--ai_genbench_dir", str(AI_GENBENCH_DIR),
    "--wildrf_dir", str(WILDRF_DIR),
    "--synthwildx_dir", str(SYNTHWILDX_DIR),
    "--bfree_viral_dir", str(BFREE_VIRAL_DIR),
    "--output_root", str(OUTPUT_ROOT),
]
subprocess.run(cmd, check=True)


In [ ]:
# ============================ CELL 5: TỔNG HỢP BẢNG PASTE-BACK ============================
import pandas as pd

print("| Run | WildRF+Viral AUC | WildRF+Viral bAcc@0.5 | WildRF+Viral ECE_cal | AIGB-Unseen AUC | AIGB-Unseen bAcc@0.5 | AIGB-Unseen ECE_cal | RealDeg-Heldout bAcc@0.5 | AIGB-Seen AUC (Guard) | Temp T | ckpt_sha256 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :--- |")
for run_dir in sorted(OUTPUT_ROOT.glob("*")):
    sum_csv = run_dir / "eval_summary.csv"
    if not sum_csv.is_file():
        continue
    s = pd.read_csv(sum_csv).set_index("subset")
    g = lambda sub, col: f"{s.loc[sub, col]:.4f}" if sub in s.index else "NA"
    t_val = f"{s['temp_T'].iloc[0]:.3f}" if len(s) else "1.000"
    sha = s["ckpt_sha256"].iloc[0] if len(s) else "NA"
    print(f"| `{run_dir.name}` | {g('wildrf_plus_viral','auc')} | {g('wildrf_plus_viral','bacc_05')} | {g('wildrf_plus_viral','ece_cal')} | {g('aigb_unseen','auc')} | {g('aigb_unseen','bacc_05')} | {g('aigb_unseen','ece_cal')} | {g('realdeg_heldout','bacc_05')} | {g('aigb_seen','auc')} | {t_val} | `{sha}` |")
